# Fertilizer Recommendation System

## Objective

Develop a machine learning model that recommends the most suitable fertilizer based on soil properties, crop type, nutrient levels, and environmental conditions. The model aims to assist in efficient nutrient management, improve crop productivity, and reduce excessive fertilizer usage.

## Input Features
Temperature
Humidity
Moisture
Soil Type
Crop Type
Nitrogen (N)
Potassium (K)
Phosphorous (P)
Output
Recommended Fertilizer

Machine Learning Task
Problem Type: Multi-Class Classification

# Step 1: Import Required Libraries

In [ ]:
# ==========================================
# Data Handling
# ==========================================

import pandas as pd
import numpy as np

# ==========================================
# Visualization
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

# ==========================================
# Machine Learning
# ==========================================

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# ==========================================
# Models
# ==========================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ==========================================
# Evaluation
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ==========================================
# Model Saving
# ==========================================

import joblib
import warnings

warnings.filterwarnings("ignore")

print("Libraries Loaded Successfully.")

# Step 2: Load & Inspect Dataset

In [ ]:
# Load Dataset

df = pd.read_csv("/kaggle/input/datasets/rohithg027/fertilizers-data/YieldCraft_Fertilizer_Dataset_v2.csv")

print("Shape:", df.shape)

df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.sample(10)

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.nunique()

In [ ]:
df["Fertilizer Name"].value_counts()

# Step 3 : Exploratory Data Analysis (EDA)

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print(f"Rows        : {df.shape[0]}")
print(f"Columns     : {df.shape[1]}")
print(f"Missing     : {df.isnull().sum().sum()}")
print(f"Duplicates  : {df.duplicated().sum()}")

display(df.head())

In [ ]:
df.dtypes

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numerical Features")
print(numerical_cols)

print("\nCategorical Features")
print(categorical_cols)

In [ ]:
df[numerical_cols].describe().T

In [ ]:
plt.figure(figsize=(15,6))

sns.countplot(
    data=df,
    x="Fertilizer Name",
    order=df["Fertilizer Name"].value_counts().index
)

plt.xticks(rotation=90)
plt.title("Distribution of Fertilizer Recommendations")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,6))

sns.countplot(
    data=df,
    x="Crop Type",
    order=sorted(df["Crop Type"].unique())
)

plt.xticks(rotation=45)
plt.title("Crop Distribution")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="Soil Type",
    order=df["Soil Type"].value_counts().index
)

plt.title("Soil Type Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x="Growth Stage",
    order=["Seedling","Vegetative","Flowering","Fruiting","Harvest"]
)

plt.title("Growth Stage Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

corr = df[numerical_cols].corr()

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")
plt.show()

In [ ]:
df[numerical_cols].hist(
    figsize=(16,12),
    bins=30
)

plt.tight_layout()
plt.show()

In [ ]:
for col in numerical_cols:

    plt.figure(figsize=(7,3))

    sns.boxplot(
        x=df[col]
    )

    plt.title(col)

    plt.show()

In [ ]:
plt.figure(figsize=(16,8))

pd.crosstab(
    df["Crop Type"],
    df["Fertilizer Name"]
).plot(
    kind="bar",
    stacked=True,
    figsize=(18,7)
)

plt.title("Crop vs Fertilizer")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,6))

pd.crosstab(
    df["Soil Type"],
    df["Fertilizer Name"]
).plot(
    kind="bar",
    stacked=True,
    figsize=(15,6)
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(3,1,figsize=(16,15))

sns.boxplot(
    data=df,
    x="Crop Type",
    y="Nitrogen",
    ax=ax[0]
)

sns.boxplot(
    data=df,
    x="Crop Type",
    y="Phosphorous",
    ax=ax[1]
)

sns.boxplot(
    data=df,
    x="Crop Type",
    y="Potassium",
    ax=ax[2]
)

for a in ax:
    a.tick_params(axis="x", rotation=90)

plt.tight_layout()

plt.show()

In [ ]:
sns.pairplot(

    df[
        [
            "Temperature",
            "Humidity",
            "Rainfall",
            "Nitrogen",
            "Phosphorous",
            "Potassium"
        ]
    ],

    corner=True

)

plt.show()

# Step 4 : Data Preprocessing & Feature Engineering

In [ ]:
# Features
X = df.drop("Fertilizer Name", axis=1)

# Target
y = df["Fertilizer Name"]

print("Feature Shape :", X.shape)
print("Target Shape  :", y.shape)

In [ ]:
numerical_features = [
    "Temperature",
    "Humidity",
    "Rainfall",
    "Moisture",
    "Soil pH",
    "Nitrogen",
    "Phosphorous",
    "Potassium"
]

categorical_features = [
    "Soil Type",
    "Crop Type",
    "Growth Stage"
]

print("Numerical Features :", numerical_features)
print()
print("Categorical Features :", categorical_features)

In [ ]:
from sklearn.preprocessing import LabelEncoder

target_encoder = LabelEncoder()

y_encoded = target_encoder.fit_transform(y)

print("Number of Classes :", len(target_encoder.classes_))
print()

print(target_encoder.classes_)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )

    ]

)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.20,

    random_state=42,

    stratify=y_encoded

)

print("Training Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

In [ ]:
train_classes = pd.Series(y_train).value_counts().sort_index()
test_classes = pd.Series(y_test).value_counts().sort_index()

comparison = pd.DataFrame({

    "Train": train_classes,

    "Test": test_classes

})

comparison.head(10)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed Training Shape :", X_train_processed.shape)
print("Processed Testing Shape  :", X_test_processed.shape)

In [ ]:
feature_names = preprocessor.get_feature_names_out()

print("Total Features After Encoding :", len(feature_names))

feature_names[:20]

In [ ]:
import os
import joblib

os.makedirs("artifacts", exist_ok=True)

joblib.dump(preprocessor, "artifacts/preprocessor.pkl")
joblib.dump(target_encoder, "artifacts/target_encoder.pkl")

print("Preprocessor Saved")
print("Target Encoder Saved")

# Step 5 : Model Benchmarking

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import time

models = {

    "Logistic Regression": LogisticRegression(max_iter=500),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        n_estimators=200
    ),

    "Extra Trees": ExtraTreesClassifier(
        random_state=42,
        n_estimators=200
    ),

    "XGBoost": XGBClassifier(

        random_state=42,

        eval_metric="mlogloss",

        use_label_encoder=False
    ),

    "LightGBM": LGBMClassifier(
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(

        random_state=42,

        verbose=False
    )

}

In [ ]:
results = []

trained_models = {}

for name, model in models.items():

    print("=" * 60)
    print(name)
    print("=" * 60)

    pipeline = Pipeline([

        ("preprocessor", preprocessor),

        ("model", model)

    ])

    start = time.time()

    pipeline.fit(X_train, y_train)

    training_time = time.time() - start

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(

        y_test,
        predictions,

        average="weighted"

    )

    recall = recall_score(

        y_test,

        predictions,

        average="weighted"

    )

    f1 = f1_score(

        y_test,

        predictions,

        average="weighted"

    )

    trained_models[name] = pipeline

    results.append({

        "Model": name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1,

        "Training Time (s)": round(training_time,2)

    })

    print("Accuracy :", round(accuracy,4))

In [ ]:
results_df = pd.DataFrame(results)

results_df.sort_values(

    by="Accuracy",

    ascending=False,

    inplace=True

)

results_df.reset_index(drop=True,inplace=True)

results_df

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(

    data=results_df,

    x="Accuracy",

    y="Model"

)

plt.xlim(0,1)

plt.title("Model Accuracy Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(

    data=results_df,

    x="F1 Score",

    y="Model"

)

plt.xlim(0,1)

plt.title("Weighted F1 Score Comparison")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(

    data=results_df,

    x="Training Time (s)",

    y="Model"

)

plt.title("Training Time Comparison")

plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

top_models = [

    "Random Forest",

    "Extra Trees",

    "XGBoost",

    "LightGBM",

    "CatBoost"

]

cv_results = []

for model_name in top_models:

    scores = cross_val_score(

        trained_models[model_name],

        X,

        y_encoded,

        cv=5,

        scoring="accuracy",

        n_jobs=-1

    )

    cv_results.append({

        "Model": model_name,

        "Mean Accuracy": scores.mean(),

        "Std": scores.std()

    })

cv_df = pd.DataFrame(cv_results)

cv_df.sort_values(

    by="Mean Accuracy",

    ascending=False
)

In [ ]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Model :", best_model_name)

best_pipeline = trained_models[best_model_name]

# Step 6 : Hyperparameter Tuning & Final Model Selection

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier

In [ ]:
param_grid = {

    "model__depth": [4, 5, 6, 7, 8, 9],

    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],

    "model__iterations": [200, 300, 500, 700],

    "model__l2_leaf_reg": [1, 3, 5, 7, 9],

    "model__border_count": [32, 64, 128, 254]

}

In [ ]:
cat_pipeline = Pipeline([

    ("preprocessor", preprocessor),

    ("model", CatBoostClassifier(

        random_state=42,

        verbose=False

    ))

])

In [ ]:
random_search = RandomizedSearchCV(

    estimator=cat_pipeline,

    param_distributions=param_grid,

    n_iter=20,

    cv=5,

    scoring="accuracy",

    verbose=2,

    random_state=42,

    n_jobs=-1

)

random_search.fit(X_train, y_train)

In [ ]:
print("Best Parameters\n")

print(random_search.best_params_)

In [ ]:
print("Best CV Accuracy :")

print(random_search.best_score_)

In [ ]:
best_model = random_search.best_estimator_

In [ ]:
predictions = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score

)

accuracy = accuracy_score(y_test, predictions)

precision = precision_score(

    y_test,

    predictions,

    average="weighted"

)

recall = recall_score(

    y_test,

    predictions,

    average="weighted"

)

f1 = f1_score(

    y_test,

    predictions,

    average="weighted"

)

print(f"Accuracy : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall : {recall:.4f}")

print(f"F1 Score : {f1:.4f}")

In [ ]:
import joblib
import os

os.makedirs("artifacts", exist_ok=True)

joblib.dump(best_model, "artifacts/fertilizer_recommendation_model.pkl")

print("Final Model Saved Successfully")

In [ ]:
import joblib
import os

os.makedirs("artifacts", exist_ok=True)

joblib.dump(best_model, "artifacts/fertilizer_recommendation_model.pkl")

print("Final Model Saved Successfully")

# Step 7 : Model Evaluation & Performance Analysis

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    predictions,
    target_names=target_encoder.classes_
))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, predictions)

print(cm)

In [ ]:
plt.figure(figsize=(14,10))

sns.heatmap(
    cm,
    cmap="Blues",
    annot=False,
    xticklabels=target_encoder.classes_,
    yticklabels=target_encoder.classes_
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")

plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()

plt.show()

In [ ]:
evaluation = pd.DataFrame({

    "Metric":[

        "Accuracy",

        "Precision",

        "Recall",

        "F1 Score"

    ],

    "Score":[

        accuracy,

        precision,

        recall,

        f1

    ]

})

evaluation

In [ ]:
plt.figure(figsize=(8,5))

sns.barplot(

    data=evaluation,

    x="Metric",

    y="Score"

)

plt.ylim(0,1)

plt.title("Model Performance Metrics")

plt.show()

In [ ]:
cat_model = best_model.named_steps["model"]

feature_names = best_model.named_steps[
    "preprocessor"
].get_feature_names_out()

importance = pd.DataFrame({

    "Feature":feature_names,

    "Importance":cat_model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

importance.head(20)

In [ ]:
plt.figure(figsize=(12,8))

top20 = importance.head(20)

sns.barplot(

    data=top20,

    x="Importance",

    y="Feature"

)

plt.title("Top 20 Important Features")

plt.show()

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(

    importance["Importance"].values

)

plt.xlabel("Feature Rank")

plt.ylabel("Importance")

plt.title("Feature Importance Distribution")

plt.grid(True)

plt.show()

In [ ]:
results = pd.DataFrame({

    "Actual":target_encoder.inverse_transform(y_test),

    "Predicted":target_encoder.inverse_transform(predictions)

})

results["Correct"] = (

    results["Actual"] == results["Predicted"]

)

results.head(20)

In [ ]:
results["Correct"].value_counts()

In [ ]:
correct = results["Correct"].sum()

incorrect = len(results) - correct

plt.figure(figsize=(6,6))

plt.pie(

    [correct, incorrect],

    labels=["Correct","Incorrect"],

    autopct="%1.1f%%",

    startangle=90

)

plt.title("Prediction Accuracy Distribution")

plt.show()

In [ ]:
misclassified = results[

    results["Correct"] == False

]

misclassified.head(20)

In [ ]:
sample_predictions = pd.DataFrame({

    "Actual":

    target_encoder.inverse_transform(y_test[:15]),

    "Predicted":

    target_encoder.inverse_transform(predictions[:15])

})

sample_predictions

# Step 8 : Deployment & Inference Pipeline

In [ ]:
import joblib

# Load trained model
model = joblib.load("artifacts/fertilizer_recommendation_model.pkl")

print("Model Loaded Successfully")

In [ ]:
target_encoder = joblib.load("artifacts/target_encoder.pkl")

print("Target Encoder Loaded")

In [ ]:
import pandas as pd

def recommend_fertilizer(
    temperature,
    humidity,
    rainfall,
    moisture,
    soil_type,
    soil_ph,
    nitrogen,
    phosphorous,
    potassium,
    crop_type,
    growth_stage
):

    input_df = pd.DataFrame({

        "Temperature": [temperature],
        "Humidity": [humidity],
        "Rainfall": [rainfall],
        "Moisture": [moisture],
        "Soil Type": [soil_type],
        "Soil pH": [soil_ph],
        "Nitrogen": [nitrogen],
        "Phosphorous": [phosphorous],
        "Potassium": [potassium],
        "Crop Type": [crop_type],
        "Growth Stage": [growth_stage]

    })

    prediction = model.predict(input_df)

    fertilizer = target_encoder.inverse_transform(prediction)[0]

    return fertilizer

In [ ]:
fertilizer = recommend_fertilizer(

    temperature=30,

    humidity=82,

    rainfall=210,

    moisture=65,

    soil_type="Loamy",

    soil_ph=6.4,

    nitrogen=55,

    phosphorous=45,

    potassium=40,

    crop_type="Rice",

    growth_stage="Vegetative"

)

print("Recommended Fertilizer :", fertilizer)

In [ ]:
import numpy as np

def predict_with_confidence(input_df):

    probabilities = model.predict_proba(input_df)

    prediction = np.argmax(probabilities)

    confidence = np.max(probabilities)

    fertilizer = target_encoder.inverse_transform([prediction])[0]

    return fertilizer, confidence

In [ ]:
sample = pd.DataFrame({

    "Temperature":[30],
    "Humidity":[82],
    "Rainfall":[210],
    "Moisture":[65],
    "Soil Type":["Loamy"],
    "Soil pH":[6.4],
    "Nitrogen":[55],
    "Phosphorous":[45],
    "Potassium":[40],
    "Crop Type":["Rice"],
    "Growth Stage":["Vegetative"]

})

fertilizer, confidence = predict_with_confidence(sample)

print("Recommended Fertilizer :", fertilizer)
print("Confidence :", round(confidence*100,2), "%")

In [ ]:
user_input = {

    "Temperature":28,
    "Humidity":75,
    "Rainfall":180,
    "Moisture":58,
    "Soil Type":"Clay",
    "Soil pH":6.5,
    "Nitrogen":70,
    "Phosphorous":40,
    "Potassium":55,
    "Crop Type":"Maize",
    "Growth Stage":"Flowering"

}

sample = pd.DataFrame([user_input])

prediction = model.predict(sample)

fertilizer = target_encoder.inverse_transform(prediction)[0]

print(fertilizer)

In [ ]:
import shutil

# Zip the existing artifacts folder
shutil.make_archive(
    "/kaggle/working/artifacts",
    "zip",
    "/kaggle/working/artifacts"
)

print("✅ Created: /kaggle/working/artifacts.zip")